# Notebook 02: Window Functions

**Phase 1 — Window Functions (PostgreSQL)**

Window functions compute across a set of rows *without* collapsing them — each
row retains its own value alongside the aggregated result.  This enables
patterns like "rank this row within its group" or "compare to the previous row"
as single-pass operations rather than multi-scan self-joins.

This notebook works through three families of window functions on the TPC-H
dataset (`sf=1`, ~1 GB, 1.5 M orders):

| Family | Functions |
|:---|:---|
| Ranking | `ROW_NUMBER`, `RANK`, `DENSE_RANK`, `NTILE` |
| Offset | `LAG`, `LEAD` |
| Aggregates with frames | `SUM OVER`, `AVG OVER`, `FIRST_VALUE`, `LAST_VALUE` |

---

## Prerequisites

Notebook 01 must have been run first (PostgreSQL seeded, Parquet files present).


In [1]:
import pathlib
import sys

import psycopg2
import pandas as pd
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from config import settings

pg = psycopg2.connect(settings.dsn)
print(f"PostgreSQL connected: {settings.POSTGRES_HOST}:{settings.POSTGRES_PORT}/{settings.POSTGRES_DB}")


def run_query(sql: str) -> pd.DataFrame:
    """Execute a SELECT statement and return results as a DataFrame."""
    with pg.cursor() as cur:
        cur.execute(sql)
        cols = [d[0] for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)


SQL_DIR = ROOT / "sql" / "window_functions"


def load_section(filename: str, section: str) -> str:
    """Extract a named section from a SQL file (delimited by '-- § name' lines)."""
    text = (SQL_DIR / filename).read_text()
    blocks: dict[str, str] = {}
    current: str | None = None
    acc: list[str] = []
    for line in text.splitlines():
        if line.startswith("-- §"):
            if current is not None:
                blocks[current] = "\n".join(acc).strip()
            current = line[4:].strip()
            acc = []
        else:
            acc.append(line)
    if current is not None:
        blocks[current] = "\n".join(acc).strip()
    if section not in blocks:
        raise KeyError(f"Section '{section}' not found in {filename}. Available: {list(blocks)}")
    return blocks[section]


PostgreSQL connected: localhost:5432/tpch


---

## 1 · Ranking Functions

**Business question:** Which customers are the biggest spenders within each nation?

Ranking without window functions requires a **self-join**: for each customer,
count how many rivals in the same nation outspent them.  At TPC-H scale
(150,000 customers × 1.5 M orders) this is a Cartesian product within each
nation partition — expensive to run and returns only one rank variant per query.


In [2]:
# Naive approach: self-join to simulate RANK()
# Rank = number of customers in the same nation with strictly higher spend + 1.
# Filtered to GERMANY for readability; still heavy at TPC-H customer scale.

naive_sql = """
WITH customer_spend AS (
    SELECT
        c.c_custkey,
        c.c_name,
        c.c_nationkey,
        n.n_name            AS nation,
        SUM(o.o_totalprice) AS total_spend
    FROM   customer c
    JOIN   orders   o ON c.c_custkey   = o.o_custkey
    JOIN   nation   n ON c.c_nationkey = n.n_nationkey
    WHERE  n.n_name = 'GERMANY'
    GROUP  BY c.c_custkey, c.c_name, c.c_nationkey, n.n_name
)
SELECT
    a.c_name,
    a.nation,
    a.total_spend,
    COUNT(b.c_custkey) + 1  AS manual_rank
FROM   customer_spend a
LEFT   JOIN customer_spend b
    ON  a.c_nationkey = b.c_nationkey
    AND b.total_spend > a.total_spend
GROUP  BY a.c_custkey, a.c_name, a.nation, a.total_spend
ORDER  BY manual_rank
LIMIT  20
"""

run_query(naive_sql)


,c_name,nation,total_spend,manual_rank
0,Customer#000056317,GERMANY,6117381.16,1
1,Customer#000100960,GERMANY,5762664.67,2
2,Customer#000047872,GERMANY,5701057.35,3
3,Customer#000044449,GERMANY,5662733.19,4
4,Customer#000112867,GERMANY,5617574.89,5
5,Customer#000014845,GERMANY,5568039.51,6
6,Customer#000054427,GERMANY,5556482.17,7
7,Customer#000075970,GERMANY,5443170.58,8
8,Customer#000016813,GERMANY,5405124.65,9
9,Customer#000075955,GERMANY,5379729.62,10


### ROW_NUMBER / RANK / DENSE_RANK

Rewritten as a single window function query — one pass, three rank variants.

The structural change: the `GROUP BY` aggregate and the window functions
coexist in the same `SELECT`.  The named `WINDOW w AS (...)` definition is
written once and referenced by each function — no repetition of
`PARTITION BY / ORDER BY`.


In [3]:
# Print the SQL for reference before executing
print(load_section("01_ranking.sql", "ranking_window"))


KeyError: "Section 'ranking_window' not found in 01_ranking.sql. Available: []"

In [ ]:
df_ranking = run_query(load_section("01_ranking.sql", "ranking_window"))
df_ranking.head(30)


### Tie-handling illustrated

A contrived 5-row example with deliberate ties makes the difference concrete.
`ROW_NUMBER` never repeats; `RANK` and `DENSE_RANK` both give the same number
to tied rows but diverge in what comes next.


In [ ]:
tie_sql = """
SELECT
    name,
    score,
    ROW_NUMBER() OVER (ORDER BY score DESC) AS row_num,
    RANK()       OVER (ORDER BY score DESC) AS rnk,
    DENSE_RANK() OVER (ORDER BY score DESC) AS dense_rnk
FROM (VALUES
    ('Alice', 100),
    ('Bob',    90),
    ('Carol',  90),
    ('Dave',   80),
    ('Eve',    80)
) AS t(name, score)
"""

run_query(tie_sql)


### NTILE(4): Revenue Quartile Segmentation

`NTILE(n)` divides all rows into `n` equal-sized buckets ordered by the window
`ORDER BY`.  Unlike `RANK`, the output is always exactly `1..n` — ties that
fall on a bucket boundary are distributed as evenly as possible.

Useful when you need exactly `n` distinct groups (e.g. "top quartile customers")
rather than a rank that can skip numbers.


In [ ]:
df_ntile = run_query(load_section("01_ranking.sql", "ntile_quartiles"))
df_ntile


---

## 2 · LAG and LEAD

**Business question:** How does each clerk's monthly revenue compare to the
previous month?  What does the following month look like?

`LAG(col)` returns the value from the **previous** row in the ordered partition.
`LEAD(col)` returns the value from the **next** row.
Both return `NULL` at the boundary rows (first and last month respectively).

The naive alternative is a self-join on a date offset:
`prev.order_month = (curr.order_month - INTERVAL '1 month')::DATE`.
This requires two full scans of the CTE and silently misses months that have
no matching prior month row.


In [ ]:
# Naive approach: self-join on a date offset to get previous month's revenue.
naive_mom_sql = """
WITH monthly_clerk AS (
    SELECT
        o_clerk,
        DATE_TRUNC('month', o_orderdate)::DATE  AS order_month,
        SUM(o_totalprice)                       AS monthly_revenue
    FROM   orders
    GROUP  BY o_clerk, DATE_TRUNC('month', o_orderdate)
)
SELECT
    curr.o_clerk,
    curr.order_month,
    curr.monthly_revenue,
    prev.monthly_revenue                             AS prev_month_revenue,
    ROUND(
        (curr.monthly_revenue - prev.monthly_revenue)::NUMERIC, 2
    )                                                AS mom_change
FROM   monthly_clerk curr
LEFT   JOIN monthly_clerk prev
    ON  curr.o_clerk    = prev.o_clerk
    AND prev.order_month = (curr.order_month - INTERVAL '1 month')::DATE
ORDER  BY o_clerk, order_month
LIMIT  20
"""

run_query(naive_mom_sql)


### LAG / LEAD with percentage change

The window version computes `LAG`, `LEAD`, month-over-month absolute change,
and percentage change in a single pass.  A second CTE (`with_lag`) materialises
the `LAG` / `LEAD` results so they can be reused in arithmetic without
repeating the window call.


In [ ]:
print(load_section("02_lag_lead.sql", "lag_lead"))


In [ ]:
df_lag = run_query(load_section("02_lag_lead.sql", "lag_lead"))
df_lag.head(20)


The monthly revenue for the first clerk in the result, plotted:


In [ ]:
clerk_id = df_lag["o_clerk"].iloc[0]
df_clerk = df_lag[df_lag["o_clerk"] == clerk_id].copy()
df_clerk["order_month"] = pd.to_datetime(df_clerk["order_month"])

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(df_clerk["order_month"], df_clerk["monthly_revenue"].astype(float),
       width=20, alpha=0.6, label="Monthly Revenue")
ax.plot(df_clerk["order_month"], df_clerk["monthly_revenue"].astype(float),
        "o-", color="C1", linewidth=1.5, markersize=4)
ax.set_title(f"Monthly Revenue — {clerk_id}")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue ($)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.legend()
plt.tight_layout()
plt.show()


---

## 3 · Running Totals and Moving Averages

This section demonstrates aggregate window functions with explicit **frame
clauses** — the `ROWS BETWEEN ... AND ...` syntax that controls exactly which
rows are included in each function call.

| Frame | Meaning |
|:---|:---|
| `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` | Start of partition to current row — running total |
| `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` | Current + 2 prior rows — 3-row sliding window |
| `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING` | Entire partition — required for `LAST_VALUE` to reach the true last row |

> **Implicit frame gotcha:** adding `ORDER BY` to a window function silently
> changes the default frame from "whole partition" to "start to current row".
> This means `SUM OVER (PARTITION BY x ORDER BY date)` is already a running
> total — but writing the frame explicitly makes the intent unambiguous.


In [ ]:
print(load_section("03_running_totals_and_moving_avg.sql", "running_total"))


In [ ]:
df_running = run_query(load_section("03_running_totals_and_moving_avg.sql", "running_total"))
df_running.head(12)


In [ ]:
print(load_section("03_running_totals_and_moving_avg.sql", "moving_avg_3m"))


In [ ]:
df_mavg = run_query(load_section("03_running_totals_and_moving_avg.sql", "moving_avg_3m"))
df_mavg.head(12)


In [ ]:
df_mavg["order_month"] = pd.to_datetime(df_mavg["order_month"])

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_mavg["order_month"], df_mavg["monthly_revenue"].astype(float),
        label="Monthly Revenue", alpha=0.4, linewidth=1, color="C0")
ax.plot(df_mavg["order_month"], df_mavg["moving_avg_3m"].astype(float),
        label="3-Month Moving Avg", linewidth=2.5, color="C1")
ax.set_title("Monthly Revenue vs 3-Month Moving Average")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue ($)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.legend()
plt.tight_layout()
plt.show()


### FIRST_VALUE / LAST_VALUE

`FIRST_VALUE` and `LAST_VALUE` return the expression from the first and last
row of the **current window frame**.

**Common gotcha with `LAST_VALUE`:** the default frame stops at the current
row, so `LAST_VALUE` returns the *current* row's value — not the partition's
last.  The frame must explicitly extend to `UNBOUNDED FOLLOWING`.


In [ ]:
print(load_section("03_running_totals_and_moving_avg.sql", "first_last_value"))


In [ ]:
df_flv = run_query(load_section("03_running_totals_and_moving_avg.sql", "first_last_value"))
df_flv.head(20)


---

## Summary

| ✅ | Concept | Function(s) |
|:---|:---|:---|
| ✅ | Rank rows within a partition | `ROW_NUMBER`, `RANK`, `DENSE_RANK` |
| ✅ | Divide into equal-sized buckets | `NTILE` |
| ✅ | Compare to adjacent rows | `LAG`, `LEAD` |
| ✅ | Cumulative aggregation | `SUM OVER (ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)` |
| ✅ | Sliding window | `AVG OVER (ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)` |
| ✅ | Anchor values across a partition | `FIRST_VALUE`, `LAST_VALUE` |

**Key insight:** every pattern above would otherwise require a self-join or
repeated subquery scan.  Window functions express the same semantics in a single
pass — more readable and consistently more efficient at scale.

**Next:** [Notebook 03 — CTEs and Recursive Queries](03_ctes_and_recursive.ipynb)
